# Polymarket On-Chain Viability Sample

This notebook mirrors the established `Polymarket_data` project at a deliberately small scale. The upstream project builds a full historical dataset by fetching Polymarket `OrderFilled` logs from Polygon, decoding the event payloads, joining CLOB token IDs to Gamma market metadata, and producing analysis-ready `trades`, `quant`, and `users` tables.

Here we run the same style of pipeline for a tiny recent block window only. The goal is not completeness; it is to prove that the data path is viable on this machine before committing to a larger backfill.

## What is copied from `Polymarket_data`

The established project's shape is:

1. Fetch market metadata from Polymarket Gamma API.
2. Fetch `OrderFilled` logs from Polygon RPC for Polymarket exchange contracts.
3. Decode event topics and ABI-encoded data into an `orderfilled` table.
4. Extract trades by identifying the non-USDC CTF asset, normalizing raw amounts by `1e6`, and joining token IDs to market metadata.
5. Produce `quant` by flipping token2 prices into a unified token1/Yes perspective.
6. Produce `users` by splitting each trade into maker and taker rows.

This notebook keeps those same concepts, but it fetches only tokens observed in the tiny on-chain sample instead of crawling the full Gamma market catalog.

## Current contract note

The bundled `Polymarket_data` repo targets the older v1 CTF exchange addresses. Polymarket's current documentation lists v2 exchange addresses, and v2 has a redesigned `OrderFilled` event with `side`, `tokenId`, `makerAmountFilled`, `takerAmountFilled`, `fee`, `builder`, and `metadata` fields.

The helper module supports both v1 and v2 decoders. The default recent-block viability sample uses v2 because current blocks are expected to contain v2 events, while v1 may be inactive after the 2026 CLOB v2 migration.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from prediction_market.polymarket_onchain import run_and_save_onchain_sample

OUTPUT_DIR = PROJECT_ROOT / "data" / "onchain_sample"

# A tiny timeframe: Polygon blocks are roughly a few seconds apart.
# Increase LOOKBACK_BLOCKS after this notebook proves the path works.
LOOKBACK_BLOCKS = 3
BATCH_SIZE = 1
MAX_EVENTS = 250

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)

## Run the small on-chain sample

This cell queries Polygon RPC for recent `OrderFilled` logs, decodes up to `MAX_EVENTS`, fetches Gamma metadata only for observed token IDs, writes raw and processed artifacts, and returns the in-memory tables.

If the public RPC endpoint is slow or empty for a block, re-run the cell or set `POLYGON_RPC_URL` to your own Polygon RPC endpoint before starting Jupyter.

In [2]:
result = run_and_save_onchain_sample(
    OUTPUT_DIR,
    lookback_blocks=LOOKBACK_BLOCKS,
    batch_size=BATCH_SIZE,
    max_events=MAX_EVENTS,
)

sample = result["sample"]
paths = result["paths"]

summary = sample["summary"]
display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

,value
retrieved_at,2026-06-06T22:09:19Z
rpc_url,https://polygon-bor-rpc.publicnode.com
start_block,88052514
end_block,88052516
lookback_blocks,3
batch_size,1
max_events,250
truncated,True
failed_ranges,[]
decoded_events,250


In [3]:
print("Raw files:")
for name, path in paths["raw"].items():
    print(f"  {name}: {path.relative_to(PROJECT_ROOT)}")

print("\nProcessed Parquet files:")
for name, path in paths["processed"].items():
    print(f"  {name}: {path.relative_to(PROJECT_ROOT)}")

print("\nCSV previews:")
for name, path in paths["latest_result"].items():
    print(f"  {name}: {path.relative_to(PROJECT_ROOT)}")

Raw files:
  summary: data/onchain_sample/raw/summary.json
  orderfilled_logs: data/onchain_sample/raw/orderfilled_logs.json
  gamma_markets: data/onchain_sample/raw/gamma_markets.json

Processed Parquet files:
  events: data/onchain_sample/processed/orderfilled.parquet
  token_mapping: data/onchain_sample/processed/market_token_mapping.parquet
  trades: data/onchain_sample/processed/trades.parquet
  quant: data/onchain_sample/processed/quant.parquet
  users: data/onchain_sample/processed/users.parquet

CSV previews:
  events: data/onchain_sample/latest_result/orderfilled.csv
  token_mapping: data/onchain_sample/latest_result/market_token_mapping.csv
  trades: data/onchain_sample/latest_result/trades.csv
  quant: data/onchain_sample/latest_result/quant.csv
  users: data/onchain_sample/latest_result/users.csv


## Raw logs and decoded `orderfilled`

Raw logs preserve the original Polygon payload. The decoded `orderfilled` table mirrors the upstream project's first processed layer: block number, transaction hash, log index, contract, maker, taker, token IDs, filled amounts, and event-specific fee fields.

In [4]:
raw_logs = sample["raw_logs"]
print(f"Raw logs captured: {len(raw_logs)}")
if raw_logs:
    print(json.dumps(raw_logs[0], indent=2, sort_keys=True)[:4000])

events = sample["events"]
display(events.drop(columns=["raw_log"]).head(20))

Raw logs captured: 250
{
  "address": "0xe111180000d2663c0091e4f400237545b87b996b",
  "blockHash": "0x552e60d688b24f2ead483b41bbfda9f4d11a3f85bb348abb8e39f7918fa2f39e",
  "blockNumber": "0x53f9322",
  "blockTimestamp": "0x6a249a8c",
  "data": "0x0000000000000000000000000000000000000000000000000000000000000000ec46d0169140e97af27f57b6471fecfc8a692eaa8c93b47691f164d5fdb3e5260000000000000000000000000000000000000000000000000000000002c779800000000000000000000000000000000000000000000000000000000002e51e90000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000",
  "logIndex": "0x13f",
  "removed": false,
  "topics": [
    "0xd543adfd945773f1a62f74f0ee55a5e3b9b1a28262980ba90b1a89f2ea84d8ee",
    "0x1f2b120d52d2bbfbd2ff8ef5886c9134719b33008407f696d97e577bbb33d8d2",
    "0x000000000000000000000000caad1b3a62d286214f94ace993a1e80e44aa255c",
    "0x00000000000000000

,transaction_hash,block_number,log_index,timestamp,datetime,contract,contract_generation,event_name,order_hash,maker,taker,maker_asset_id,taker_asset_id,side,side_label,token_id,maker_amount_filled,taker_amount_filled,maker_fee,taker_fee,protocol_fee,fee,builder,metadata
0,0x8d8f5f7d15e8d55860604830df7edf1af816bdaa21ea95036f9c0e7d16933218,88052514,319,1780783756,2026-06-06T22:09:16Z,CTF_EXCHANGE_V2,v2,OrderFilled,0x1f2b120d52d2bbfbd2ff8ef5886c9134719b33008407f696d97e577bbb33d8d2,0xcaad1b3a62d286214f94ace993a1e80e44aa255c,0xfd16bdd047ecdb6c006a9f159c14c8bbb7462c91,0,106870947731841299226931869706163862462989258483960759258537831115103115928870,0,BUY,106870947731841299226931869706163862462989258483960759258537831115103115928870,46627200,48570000,0,0,0,0,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
1,0x8d8f5f7d15e8d55860604830df7edf1af816bdaa21ea95036f9c0e7d16933218,88052514,322,1780783756,2026-06-06T22:09:16Z,CTF_EXCHANGE_V2,v2,OrderFilled,0x1b4d4f7be3306b43a54630de46eb5b01dc1c55d3676c887fb955624e658e3823,0xfd16bdd047ecdb6c006a9f159c14c8bbb7462c91,0xe111180000d2663c0091e4f400237545b87b996b,106870947731841299226931869706163862462989258483960759258537831115103115928870,0,1,SELL,106870947731841299226931869706163862462989258483960759258537831115103115928870,48570000,46627200,0,0,0,130550,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
2,0x75e9c920a369d446b4a8d350d5b1c25b1dc63c39304fa9bd607bf81571f1a766,88052514,328,1780783756,2026-06-06T22:09:16Z,CTF_EXCHANGE_V2,v2,OrderFilled,0x4be39b32fea472c4cff5378bde1493c457a8793df7a0292a09ba4fd86046b6fa,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,0xdc5cb48aa552995edb952bf2b0434df01e20808a,106870947731841299226931869706163862462989258483960759258537831115103115928870,0,1,SELL,106870947731841299226931869706163862462989258483960759258537831115103115928870,2000000,1940000,0,0,0,0,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
3,0x75e9c920a369d446b4a8d350d5b1c25b1dc63c39304fa9bd607bf81571f1a766,88052514,330,1780783756,2026-06-06T22:09:16Z,CTF_EXCHANGE_V2,v2,OrderFilled,0x72685190475bb1cde27664f511be30d53b2480c98603b0bc96565de5351a6853,0xdc5cb48aa552995edb952bf2b0434df01e20808a,0xe111180000d2663c0091e4f400237545b87b996b,0,106870947731841299226931869706163862462989258483960759258537831115103115928870,0,BUY,106870947731841299226931869706163862462989258483960759258537831115103115928870,1940000,2000000,0,0,0,4070,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
4,0xd8b15166989fe01858f90a81798ae183ee70a25c620197b18397c5281b30005a,88052514,336,1780783756,2026-06-06T22:09:16Z,CTF_EXCHANGE_V2,v2,OrderFilled,0x4be39b32fea472c4cff5378bde1493c457a8793df7a0292a09ba4fd86046b6fa,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,0x16bc7faccdb6dedd07d47333a6f06fef635dd23a,106870947731841299226931869706163862462989258483960759258537831115103115928870,0,1,SELL,106870947731841299226931869706163862462989258483960759258537831115103115928870,22500000,21825000,0,0,0,0,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
5,0xd8b15166989fe01858f90a81798ae183ee70a25c620197b18397c5281b30005a,88052514,338,1780783756,2026-06-06T22:09:16Z,CTF_EXCHANGE_V2,v2,OrderFilled,0x3e2309f4eaa78cd7789f991f936ff6b8de3465529c3fc8777d13c71b02e79aab,0x16bc7faccdb6dedd07d47333a6f06fef635dd23a,0xe111180000d2663c0091e4f400237545b87b996b,0,106870947731841299226931869706163862462989258483960759258537831115103115928870,0,BUY,106870947731841299226931869706163862462989258483960759258537831115103115928870,21825000,22500000,0,0,0,45830,0x0000000000000000000000000000000000000000000000000000000000000000,0x00000000000000000000000000000000000000

## Gamma token mapping

The full upstream project fetches and maintains a complete `markets.parquet`. This viability sample does a smaller join: it extracts token IDs from observed logs, asks Gamma for only those token IDs, and builds `token_id -> market` mappings for the sample.

In [5]:
token_mapping = sample["token_mapping"]
display(token_mapping.drop(columns=["raw_payload"]).head(30))

unmatched_tokens = sorted(set(events["token_id"].dropna().astype(str)) - set(token_mapping["token_id"].dropna().astype(str))) if not events.empty else []
print(f"Observed unique token IDs: {events['token_id'].nunique() if not events.empty else 0}")
print(f"Mapped token IDs: {token_mapping['token_id'].nunique() if not token_mapping.empty else 0}")
print(f"Unmatched observed token IDs: {len(unmatched_tokens)}")
if unmatched_tokens:
    print(unmatched_tokens[:10])

,token_id,market_id,condition_id,side,answer,question,event_id,event_slug,event_title,slug,active,closed,end_date
0,19394319214140646671941741664381705243202267892747350778544673112122629257973,2438947,0xd527d369e0f5d7b31f78a22e0ccb40fd0475f1b5c638b4f32a6cfe46ad88f690,token1,Up,"XRP Up or Down - June 6, 6PM ET",559980,xrp-up-or-down-june-6-2026-6pm-et,"XRP Up or Down - June 6, 6PM ET",xrp-up-or-down-june-6-2026-6pm-et,True,False,2026-06-06T23:00:00Z
1,100511062230478552212135043736276339188083946001738935434341228273920332914741,2438947,0xd527d369e0f5d7b31f78a22e0ccb40fd0475f1b5c638b4f32a6cfe46ad88f690,token2,Down,"XRP Up or Down - June 6, 6PM ET",559980,xrp-up-or-down-june-6-2026-6pm-et,"XRP Up or Down - June 6, 6PM ET",xrp-up-or-down-june-6-2026-6pm-et,True,False,2026-06-06T23:00:00Z
2,31164334329147454586598351177849035944989772929947123529371470108985530543953,2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,token1,Up,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",btc-updown-5m-1780783500,True,False,2026-06-06T22:10:00Z
3,106870947731841299226931869706163862462989258483960759258537831115103115928870,2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,token2,Down,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",btc-updown-5m-1780783500,True,False,2026-06-06T22:10:00Z
4,11709025336479019570959926169364154508096878905561934832983242232548084043273,2448222,0xdcc76b249412818c5336caf58aadfaf1ec1f8216e772d43936eec0a54e8cb96e,token1,Up,"XRP Up or Down - June 6, 6:05PM-6:10PM ET",563573,xrp-updown-5m-1780783500,"XRP Up or Down - June 6, 6:05PM-6:10PM ET",xrp-updown-5m-1780783500,True,False,2026-06-06T22:10:00Z
5,107588686919681972319972959473167522533909429810342062111677120152968428287974,2448222,0xdcc76b249412818c5336caf58aadfaf1ec1f8216e772d43936eec0a54e8cb96e,token2,Down,"XRP Up or Down - June 6, 6:05PM-6:10PM ET",563573,xrp-updown-5m-1780783500,"XRP Up or Down - June 6, 6:05PM-6:10PM ET",xrp-updown-5m-1780783500,True,False,2026-06-06T22:10:00Z
6,77604671816165649880193320279872939595250175691723251969237094641211185350049,2456148,0xaee270f117ab24d9a865def58474583df25dbe2870af4dfa734968397fc45322,token1,Yes,Will the highest temperature in Paris be 23°C on June 8?,566654,highest-temperature-in-paris-on-june-8-2026,Highest temperature in Paris on June 8?,highest-temperature-in-paris-on-june-8-2026-23c,True,False,2026-06-08T12:00:00Z
7,108184724425608771735828556185909553446466947170307951362782981055498577335128,2456148,0xaee270f117ab24d9a865def58474583df25dbe2870af4dfa734968397fc45322,token2,No,Will the highest temperature in Paris be 23°C on June 8?,566654,highest-temperature-in-paris-on-june-8-2026,Highest temperature in Paris on June 8?,highest-temperature-in-paris-on-june-8-2026-23c,True,False,2026-06-08T12:00:00Z
8,76339066435834419633431563584009610571270345858031834833226699514847391136686,2305292,0x6e8e96058b8cd0047156ba7c2db0839926b8be52ad9903948ee14f5c2166442a,token1,Over,Cabo Verde vs. Bermuda: O/U 4.5,503126,fif-cvi-ber-2026-06-06-more-markets,Cabo Verde vs. Bermuda - More Markets,fif-cvi-ber-2026-06-06-total-4pt5,True,False,2026-06-06T20:00:00Z
9,108731524654123509997510843344654240937427302880300958042509229562248362996002,2305292,0x6e8e96058b8cd0047156ba7c2db0839926b8be52ad9903948ee14f5c2166442a,token2,Under,Cabo Verde vs. Bermuda: O/U 4.5,503126,fif-cvi-ber-2026-06-06-more-markets,Cabo Verde vs. Bermuda - More Markets,fif-cvi-ber-2026-06-06-total-4pt5,True,False,2026-06-06T20:00:00Z


Observed unique token IDs: 59
Mapped token IDs: 66
Unmatched observed token IDs: 0


## Trades

The `trades` table follows the established project's extraction logic. The non-USDC asset is the outcome token. Raw amounts are divided by `1e6`, and price is `USDC amount / token amount`.

For v2 events, `side=BUY` means the maker is buying `tokenId` with collateral; `side=SELL` means the maker is selling `tokenId` for collateral. The notebook preserves `builder` and `metadata` from v2 events as extra attribution fields.

In [6]:
trades = sample["trades"]
display(trades.head(30))

if not trades.empty:
    display(
        trades.groupby(["contract_generation", "contract", "maker_direction", "nonusdc_side"], dropna=False)
        .agg(rows=("transaction_hash", "count"), usd_amount=("usd_amount", "sum"), avg_price=("price", "mean"))
        .reset_index()
        .sort_values("rows", ascending=False)
    )

,timestamp,datetime,block_number,transaction_hash,log_index,contract,contract_generation,event_id,event_slug,event_title,market_id,condition_id,question,nonusdc_side,answer,maker,taker,maker_asset,taker_asset,maker_direction,taker_direction,price,usd_amount,token_amount,fee_amount,asset_id,order_hash,builder,metadata
0,1780783756,2026-06-06T22:09:16Z,88052514,0x8d8f5f7d15e8d55860604830df7edf1af816bdaa21ea95036f9c0e7d16933218,319,CTF_EXCHANGE_V2,v2,563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",token2,Down,0xcaad1b3a62d286214f94ace993a1e80e44aa255c,0xfd16bdd047ecdb6c006a9f159c14c8bbb7462c91,USDC,token2,BUY,SELL,0.960,46.627200,48.570000,0.00000,106870947731841299226931869706163862462989258483960759258537831115103115928870,0x1f2b120d52d2bbfbd2ff8ef5886c9134719b33008407f696d97e577bbb33d8d2,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
1,1780783756,2026-06-06T22:09:16Z,88052514,0x8d8f5f7d15e8d55860604830df7edf1af816bdaa21ea95036f9c0e7d16933218,322,CTF_EXCHANGE_V2,v2,563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",token2,Down,0xfd16bdd047ecdb6c006a9f159c14c8bbb7462c91,0xe111180000d2663c0091e4f400237545b87b996b,token2,USDC,SELL,BUY,0.960,46.627200,48.570000,0.13055,106870947731841299226931869706163862462989258483960759258537831115103115928870,0x1b4d4f7be3306b43a54630de46eb5b01dc1c55d3676c887fb955624e658e3823,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
2,1780783756,2026-06-06T22:09:16Z,88052514,0x75e9c920a369d446b4a8d350d5b1c25b1dc63c39304fa9bd607bf81571f1a766,328,CTF_EXCHANGE_V2,v2,563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",token2,Down,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,0xdc5cb48aa552995edb952bf2b0434df01e20808a,token2,USDC,SELL,BUY,0.970,1.940000,2.000000,0.00000,106870947731841299226931869706163862462989258483960759258537831115103115928870,0x4be39b32fea472c4cff5378bde1493c457a8793df7a0292a09ba4fd86046b6fa,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
3,1780783756,2026-06-06T22:09:16Z,88052514,0x75e9c920a369d446b4a8d350d5b1c25b1dc63c39304fa9bd607bf81571f1a766,330,CTF_EXCHANGE_V2,v2,563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",token2,Down,0xdc5cb48aa552995edb952bf2b0434df01e20808a,0xe111180000d2663c0091e4f400237545b87b996b,USDC,token2,BUY,SELL,0.970,1.940000,2.000000,0.00407,106870947731841299226931869706163862462989258483960759258537831115103115928870,0x72685190475bb1cde27664f511be30d53b2480c98603b0bc96565de5351a6853,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
4,1780783756,2026-06-06T22:09:16Z,88052514,0xd8b15166989fe01858f90a81798ae183ee70a25c620197b18397c5281b30005a,336,CTF_EXCHANGE_V2,v2,563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",token2,Down,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,0x16bc7faccdb6dedd07d47333a6f06fef635dd23a,token2,USDC,SELL,BUY,0.970,21.825000,22.500000,0.00000,106870947731841299226931869706163862462989258483960759258537831115103115928870,0x4be39b32fea472c4cff5378bde1493c457a8793

,contract_generation,contract,maker_direction,nonusdc_side,rows,usd_amount,avg_price
1,v2,CTF_EXCHANGE_V2,BUY,token2,99,3008.392592,0.858606
0,v2,CTF_EXCHANGE_V2,BUY,token1,52,956.625227,0.239545
3,v2,CTF_EXCHANGE_V2,SELL,token2,52,1260.856602,0.959558
4,v2,NEGRISK_CTF_EXCHANGE_V2,BUY,token1,15,357.093163,0.323185
2,v2,CTF_EXCHANGE_V2,SELL,token1,14,319.911289,0.189143
5,v2,NEGRISK_CTF_EXCHANGE_V2,BUY,token2,13,183.154780,0.765590
6,v2,NEGRISK_CTF_EXCHANGE_V2,SELL,token1,4,335.354740,0.301250
7,v2,NEGRISK_CTF_EXCHANGE_V2,SELL,token2,1,0.408360,0.996000


## Quant view

`quant` removes rows where the taker is an exchange contract intermediary and flips token2 prices into token1 perspective. After this step, `price` is comparable across binary markets as the token1/Yes-side price.

In [7]:
quant = sample["quant"]
display(quant.head(30))

if not quant.empty:
    display(
        quant.groupby(["event_slug", "market_id"], dropna=False)
        .agg(rows=("transaction_hash", "count"), usd_amount=("usd_amount", "sum"), min_price=("price", "min"), max_price=("price", "max"))
        .reset_index()
        .sort_values("rows", ascending=False)
        .head(20)
    )

,timestamp,datetime,block_number,transaction_hash,log_index,contract,contract_generation,event_id,event_slug,event_title,market_id,condition_id,question,nonusdc_side,answer,maker,taker,maker_asset,taker_asset,maker_direction,taker_direction,price,usd_amount,token_amount,fee_amount,asset_id,order_hash,builder,metadata
0,1780783756,2026-06-06T22:09:16Z,88052514,0x8d8f5f7d15e8d55860604830df7edf1af816bdaa21ea95036f9c0e7d16933218,319,CTF_EXCHANGE_V2,v2,563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",token1,Down,0xcaad1b3a62d286214f94ace993a1e80e44aa255c,0xfd16bdd047ecdb6c006a9f159c14c8bbb7462c91,USDC,token2,BUY,SELL,0.040,46.627200,48.570000,0.0,106870947731841299226931869706163862462989258483960759258537831115103115928870,0x1f2b120d52d2bbfbd2ff8ef5886c9134719b33008407f696d97e577bbb33d8d2,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
1,1780783756,2026-06-06T22:09:16Z,88052514,0x75e9c920a369d446b4a8d350d5b1c25b1dc63c39304fa9bd607bf81571f1a766,328,CTF_EXCHANGE_V2,v2,563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",token1,Down,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,0xdc5cb48aa552995edb952bf2b0434df01e20808a,token2,USDC,SELL,BUY,0.030,1.940000,2.000000,0.0,106870947731841299226931869706163862462989258483960759258537831115103115928870,0x4be39b32fea472c4cff5378bde1493c457a8793df7a0292a09ba4fd86046b6fa,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
2,1780783756,2026-06-06T22:09:16Z,88052514,0xd8b15166989fe01858f90a81798ae183ee70a25c620197b18397c5281b30005a,336,CTF_EXCHANGE_V2,v2,563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",token1,Down,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,0x16bc7faccdb6dedd07d47333a6f06fef635dd23a,token2,USDC,SELL,BUY,0.030,21.825000,22.500000,0.0,106870947731841299226931869706163862462989258483960759258537831115103115928870,0x4be39b32fea472c4cff5378bde1493c457a8793df7a0292a09ba4fd86046b6fa,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
3,1780783756,2026-06-06T22:09:16Z,88052514,0xe1db1908adb3bd175eb4cdc15549f31b680fc6b9f366e672ed97e8a2c0460786,344,CTF_EXCHANGE_V2,v2,563572,eth-updown-5m-1780783500,"Ethereum Up or Down - June 6, 6:05PM-6:10PM ET",2448220,0x1eb6352998233b4a191a8d30ac54e6fe73f07992d4b47d2085b2e0511eedfa40,"Ethereum Up or Down - June 6, 6:05PM-6:10PM ET",token1,Up,0x98da26e8fa92fc7973af6f95baf5efc4e2493b56,0x674887d1ac838099a48b629dff53f25b7b87ee08,USDC,token1,BUY,SELL,0.100,0.430000,4.300000,0.0,114452155101132071341035941016658939783020858202809214057500414093543091573082,0x04e8f45e8ed3662347b2b8686c3ed9dfad1b6d247b70795d05bcc0ca5580d51c,0x0000000000000000000000000000000000000000000000000000000000000000,0x0000000000000000000000000000000000000000000000000000000000000000
4,1780783756,2026-06-06T22:09:16Z,88052514,0x5ac0f5ea6c01482e4aacd7f2259ca54727ee7b98ee9fd048e0694fbdba64024a,353,CTF_EXCHANGE_V2,v2,563571,btc-updown-5m-1780783500,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,"Bitcoin Up or Down - June 6, 6:05PM-6:10PM ET",token1,Down,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,0xf898c9739a8b6ae576f4efcadf7241a64b49fffc,token2,USDC,SELL,BUY,0.030,58.200000,60.000000,0.0,106870947731841299226931869706163862462989258483960759258537831115103115928870,0x4be39b32fea472c4cff5378bde1493c457a8793df7a0292a09ba4fd8604

,event_slug,market_id,rows,usd_amount,min_price,max_price
4,btc-updown-5m-1780783500,2448221,59,1776.516063,0.030,0.040
3,btc-updown-15m-1780783200,2448208,12,5.295000,0.050,0.050
8,eth-updown-5m-1780783500,2448220,10,3.124509,0.090,0.100
17,lol-fly-tl2-2026-06-06,2407743,8,611.795900,0.430,0.440
14,highest-temperature-in-paris-on-june-8-2026,2456146,8,22.210000,0.210,0.250
15,highest-temperature-in-paris-on-june-8-2026,2456148,7,14.877048,0.270,0.280
30,xrp-up-or-down-june-6-2026-6pm-et,2438947,4,26.395700,0.260,0.270
32,xrp-updown-5m-1780783800,2448231,3,5.672000,0.520,0.540
24,will-alberta-join-the-us,1345724,2,7.991400,0.047,0.047
22,ufc-bru-ed-2026-06-06,2280037,2,174.857142,0.510,0.510


## Users view

`users` expands each cleaned trade into two participant rows: one for maker and one for taker. Sells are represented as negative token amounts, and direction is normalized to `BUY`, matching the established project's user-level convention.

In [8]:
users = sample["users"]
display(users.head(40))

if not users.empty:
    display(
        users.groupby(["user", "role"], dropna=False)
        .agg(rows=("transaction_hash", "count"), net_tokens=("token_amount", "sum"), usd_amount=("usd_amount", "sum"))
        .reset_index()
        .sort_values("rows", ascending=False)
        .head(20)
    )

,timestamp,datetime,block_number,transaction_hash,event_id,market_id,condition_id,user,role,price,token_amount,usd_amount
0,1780783756,2026-06-06T22:09:16Z,88052514,0x8d8f5f7d15e8d55860604830df7edf1af816bdaa21ea95036f9c0e7d16933218,563571,2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,0xcaad1b3a62d286214f94ace993a1e80e44aa255c,maker,0.040,48.570000,46.627200
1,1780783756,2026-06-06T22:09:16Z,88052514,0x8d8f5f7d15e8d55860604830df7edf1af816bdaa21ea95036f9c0e7d16933218,563571,2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,0xfd16bdd047ecdb6c006a9f159c14c8bbb7462c91,taker,0.040,-48.570000,46.627200
2,1780783756,2026-06-06T22:09:16Z,88052514,0x75e9c920a369d446b4a8d350d5b1c25b1dc63c39304fa9bd607bf81571f1a766,563571,2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,maker,0.030,-2.000000,1.940000
3,1780783756,2026-06-06T22:09:16Z,88052514,0x75e9c920a369d446b4a8d350d5b1c25b1dc63c39304fa9bd607bf81571f1a766,563571,2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,0xdc5cb48aa552995edb952bf2b0434df01e20808a,taker,0.030,2.000000,1.940000
4,1780783756,2026-06-06T22:09:16Z,88052514,0xd8b15166989fe01858f90a81798ae183ee70a25c620197b18397c5281b30005a,563571,2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,maker,0.030,-22.500000,21.825000
5,1780783756,2026-06-06T22:09:16Z,88052514,0xd8b15166989fe01858f90a81798ae183ee70a25c620197b18397c5281b30005a,563571,2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,0x16bc7faccdb6dedd07d47333a6f06fef635dd23a,taker,0.030,22.500000,21.825000
6,1780783756,2026-06-06T22:09:16Z,88052514,0xe1db1908adb3bd175eb4cdc15549f31b680fc6b9f366e672ed97e8a2c0460786,563572,2448220,0x1eb6352998233b4a191a8d30ac54e6fe73f07992d4b47d2085b2e0511eedfa40,0x98da26e8fa92fc7973af6f95baf5efc4e2493b56,maker,0.100,4.300000,0.430000
7,1780783756,2026-06-06T22:09:16Z,88052514,0xe1db1908adb3bd175eb4cdc15549f31b680fc6b9f366e672ed97e8a2c0460786,563572,2448220,0x1eb6352998233b4a191a8d30ac54e6fe73f07992d4b47d2085b2e0511eedfa40,0x674887d1ac838099a48b629dff53f25b7b87ee08,taker,0.100,-4.300000,0.430000
8,1780783756,2026-06-06T22:09:16Z,88052514,0x5ac0f5ea6c01482e4aacd7f2259ca54727ee7b98ee9fd048e0694fbdba64024a,563571,2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,maker,0.030,-60.000000,58.200000
9,1780783756,2026-06-06T22:09:16Z,88052514,0x5ac0f5ea6c01482e4aacd7f2259ca54727ee7b98ee9fd048e0694fbdba64024a,563571,2448221,0xb862d8ac42449912b6a645b041aca52131e6cc8081e18f4906452af90a8bb78b,0xf898c9739a8b6ae576f4efcadf7241a64b49fffc,taker,0.030,60.000000,58.200000


,user,role,rows,net_tokens,usd_amount
132,0xd594c3f395c3955b8b06aa5fb428b514bbf29d1b,maker,44,-1051.019457,1019.488863
114,0xb41c082290933df8904699cf67895d60ffd95bd3,taker,13,-625.000000,600.000000
150,0xee55214ee3a9ee22a404663c76ca832577df7b04,maker,5,14.983320,1.498332
12,0x16bc7faccdb6dedd07d47333a6f06fef635dd23a,taker,4,73.790000,71.576300
138,0xdcef920c3c532ae6f0e195c86bafb6fd4894d05c,taker,4,-20.000000,1.000000
117,0xbf17c1643131948cc5aecb890e915b124e7540be,maker,4,11.111977,1.000078
65,0x6a8d1709bfb718d8555d315a983c4816278350f9,taker,4,-16.840000,11.675200
70,0x710cad178b1e6fec99981fd7ff48fc05c7bd78ea,taker,4,299.750000,550.140000
98,0x9a960907c9afdb060e480e0a95eef0406a41cf5c,taker,4,-25.858629,12.918630
143,0xe9076a87c5ed90ef16e6fe6529c943baeca0cff6,taker,3,-3.645813,7.567775


## Viability readout

This small run is viable if the summary shows nonzero decoded events, nonzero trades, and at least some Gamma token mappings. A few unmatched tokens can happen in a tiny live sample or when Gamma does not return metadata for a token query, but a high unmatched rate would be the next thing to investigate before scaling.

Recommended next scale-up after this passes: increase `LOOKBACK_BLOCKS` gradually, keep `BATCH_SIZE` small, raise `MAX_EVENTS`, and write partitioned session files rather than repeatedly overwriting this sample folder.